In [1]:
from selenium import webdriver
from selenium.webdriver import Chrome, ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import  WebDriverWait
import pandas as pd 
import time, threading, os, requests, ast

In [2]:
def scarpe(path):
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()


    try:
        login = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/a[text()="Login"]'))
        )
        login.click()
    except:
        print("no login")

    try:
        user_name = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.ID, 'username'))
        )
        user_name.send_keys("fourbrotherstrading@icloud.com")
    except Exception as e:
        print("Username error", e)

    try:
        password = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.ID, 'password'))
        )
        password.send_keys("Muhssan7865")
    except Exception as e:
        print("Password error", e)

    try:
        check = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.XPATH, './/button[text()="Sign in"]'))
        )
        check.click()
    except:
        print("Sign-in button not found")


    try:
        cars = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/span[@class="sort-page"]'))
        ).text.strip()
        total_cars = int(cars.split(" ")[4])
        print(f"{total_cars} cars found")
    except Exception as e:
        print("Cannot read total cars", e)
        total_cars = 0

    car_count = 0

    while car_count < total_cars:

        try:
            page_cars = WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
            )

            for i in range(len(page_cars)):
                if car_count >= total_cars:
                    break

       
                page_cars = WebDriverWait(driver, 5).until(
                    EC.presence_of_all_elements_located((By.XPATH, './/img[@class="card-img-top"]'))
                )

                driver.execute_script("arguments[0].scrollIntoView();", page_cars[i])
                time.sleep(1)
                page_cars[i].click()


                try:
                    reg_el = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.XPATH, './/span[@class="pill-item pill-item-reg"]'))
                    )
                    reg_number = reg_el.text.strip().replace("/", "_").replace(" ", "_")

                    if not os.path.exists("html"):
                        os.makedirs("html")

                    filename = f"html/{reg_number}.html"
                    with open(filename, "w", encoding="utf-8") as f:
                        f.write(driver.page_source)

                    print(f"✔ Saved HTML: {filename}")

                except Exception as e:
                    print("HTML save error", e)

                car_count += 1
                driver.back()
                time.sleep(1)


            try:
                next_btn = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.XPATH, '//li[@class="page-item page-item-arrow page-item-arrow-next"]/a'))
                )
                next_href = next_btn.get_attribute("href")

                if next_href:
                    print(f"➡ Next page: {next_href}")
                    driver.get(next_href)
                else:
                    print("No more pages")
                    break

            except:
                print("Pagination finished")
                break

        except:
            print("No more car links")
            break

    driver.quit()


# RUN
path = 'https://www.protruckauctions.co.uk/auction/468'
scarpe(path)


48 cars found
✔ Saved HTML: html/HV11FMF.html
✔ Saved HTML: html/MT70NXE.html
✔ Saved HTML: html/PK20LCO.html
✔ Saved HTML: html/VA68ACU.html
✔ Saved HTML: html/ND22OYL.html
✔ Saved HTML: html/ND22MXS.html
✔ Saved HTML: html/ND21HHV.html
✔ Saved HTML: html/NJ69HPA.html
✔ Saved HTML: html/NL18FUF.html
✔ Saved HTML: html/CE70LTV.html
✔ Saved HTML: html/YS70MUY.html
✔ Saved HTML: html/LM74FJU.html
✔ Saved HTML: html/LM74FHD.html
✔ Saved HTML: html/DE21VLG.html
✔ Saved HTML: html/DE21VJL.html
✔ Saved HTML: html/BJ22YON.html
✔ Saved HTML: html/FL18ENO.html
✔ Saved HTML: html/HV74KGU.html
✔ Saved HTML: html/WM73VFC.html
✔ Saved HTML: html/WM73VEH.html
✔ Saved HTML: html/WM73OZW.html
✔ Saved HTML: html/WM73VHU.html
✔ Saved HTML: html/PY74DHA.html
✔ Saved HTML: html/MW17UNV.html
✔ Saved HTML: html/BK23XDF.html
✔ Saved HTML: html/KX13JXC.html
✔ Saved HTML: html/KX65KYF.html
✔ Saved HTML: html/YA13UAV.html
✔ Saved HTML: html/FD65ORA.html
✔ Saved HTML: html/FJ20UZW.html
➡ Next page: https://www.p

In [ ]:
import os,re,json
import csv
from bs4 import BeautifulSoup

def get_base_folder_info():
    folder_name = os.path.basename(os.getcwd())

    parts = folder_name.split("-")
    if not parts or not parts[0].isdigit():
        return None, None

    sheet_id = parts[0]
    name_parts = parts[1:]
    if name_parts and name_parts[0].isdigit():
        name_parts = name_parts[1:]

    auction_name = "-".join(name_parts).strip()

    return sheet_id, auction_name

def extract_details(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    result = {}

  
    details_ul = soup.find("ul", class_="details-list")
    if details_ul:
        li_items = details_ul.find_all("li", class_="detail-item")
        for li in li_items:
            key_el = li.find("span")
            value_el = li.find("strong")
            if key_el and value_el:
                key = key_el.get_text(strip=True)
                value = value_el.get_text(strip=True)
                result[key] = value


    features_p = soup.find("p", class_="mt-4")
    if features_p:
        features_text = features_p.get_text(" ", strip=True)
        result["Features"] = features_text

    return result

def extract_additional_info(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    result = {}

    details_ul = soup.find("ul", class_="tab-grid details-list")
    if details_ul:
        li_items = details_ul.find_all("li", class_="detail-item")
        for li in li_items:
            key_el = li.find("span")
            value_el = li.find("strong")
            if key_el and value_el:
                key = key_el.get_text(strip=True)
                value = value_el.get_text(strip=True)
                result[key] = value


    if result:
        return {"information": result}
    else:
        return {"information": {}}


def extract_manual_keys():
    folder = "html"
    output_file = "protruck_data.csv"

    keys = ["Title",
            "Auction Name",
            "Sheet id",
            "Auction House",
            "Make",
            "Model",
            "Variant",
            "Lot", 
            "Reg", 
            "Start Time", 
            "Start Date",
            "D.O.R",
            "Fuel Type",
            "Former Keepers",
            "Transmission",
            "Colour",
            "MOT Expiry Date",
            "Keys",
            "VAT Status",
            "V5",
            "CAP Clean",
            "CAP Average",
            "CAP Below",
            "CC",
            "Mileage",
            "Mileage Warranted",
            "MOT Due",
            "Body Type",
            "Additional information",
            "General Condition",
            "Tyres Condition",
            "Inspection Report",
            "Images",
            "Damaged_images",
            "Damage_details",
            ]  

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}

            reg_el = soup.find("span", class_="pill-item pill-item-reg")
            reg = reg_el.get_text(strip=True) if reg_el else ""
            pattern = re.compile(r'^[A-Z]{1,3}[0-9]{1,3}[A-Z]{1,3}$', re.I)
            if not pattern.match(reg):
                print(f"❌ Not valid: {reg} → Deleting file {file}")
                os.remove(file_path)
                continue
            else:
                row["Reg"] = reg
                lot_el = soup.find("span", class_="pill-item pill-item-lot")
                row["Lot"] = lot_el.get_text(strip=True).replace("Lot", "").strip() if lot_el else ""

                title_tag = soup.find("h2", class_="title-h2")
                row["Title"] = title_tag.get_text(strip=True) if title_tag else ""
                row["Make"] = row["Title"] 

                model_tag = soup.find("p", class_="title-sub")
                row["Model"] = model_tag.get_text(strip=True) if model_tag else ""

                variant_tag = soup.find("p", class_="title-sub title-sub-2")
                row["Variant"] = variant_tag.get_text(strip=True) if variant_tag else ""

                time_el = soup.find("span", class_="pill-item pill-item-time")
                if time_el:
                    time_text = time_el.get_text(strip=True)
                    parts = time_text.split(" ")
                    row["Start Time"] = parts[0] if len(parts) >= 1 else ""
                    row["Start Date"] = parts[1] if len(parts) >= 2 else ""
                else:
                    row["Start Time"] = ""
                    row["Start Date"] = ""


                findElement = extract_details(html_content)
                mot =  findElement.get("MOT", "")
                row["D.O.R"] = findElement.get("Registered", "")
                row["Fuel Type"] = findElement.get("Fuel", "")
                row["Former Keepers"] = findElement.get("Former Keepers", "")
                row["Transmission"] = findElement.get("Transmission", "")
                row["Colour"] = findElement.get("Colour", "")
                row["MOT Expiry Date"] =mot if mot != "Expired" else " "
                row["Keys"] = findElement.get("Keys", "")
                row["VAT Status"] = findElement.get("VAT", "")
                row["V5"] = findElement.get("V5", "")
                row["CAP Clean"] = findElement.get("CAP Clean", "")
                row["CAP Below"] = findElement.get("CAP Below", "")
                row["CAP Average"] = findElement.get("CAP Average", "")
                
                cc_raw = findElement.get("CC", "")
                try:
                    cc_val = round(int(cc_raw) / 10000, 1) if cc_raw else ""
                except:
                    cc_val = ""
                row["CC"] = cc_val


                warranted_raw = findElement.get("Miles Warranted", "")
                if warranted_raw.strip().lower() == "not warranted":
                    row["Mileage Warranted"] = "NO"
                elif warranted_raw:
                    row["Mileage Warranted"] = "Warranted"
                else:
                    row["Mileage Warranted"] = ""

    
                miles_raw = findElement.get("Miles", "")
                miles_val = ""
                if miles_raw:
                    miles_clean = miles_raw.replace(",", "").replace("Km", "").strip()

                    if miles_clean.replace(".", "").isdigit():  
                        miles_val = int(float(miles_clean)) 
                    else:
                        miles_val = ""  
                row["Mileage"] = miles_val
        
                extractAdditional = extract_additional_info(html_content)
                info = extractAdditional.get("information", {})
                motDue = info.get("MOT Due", "")
                row["MOT Due"] = motDue if motDue != "Expired" else " "
                row["Body Type"] = info.get("Body Style", "")
                row["Additional information"] = extractAdditional

                ul = soup.find("ul", class_="details-list")
                data = {}
                for li in ul.find_all("li"):
                    span = li.find("span")
                    strong = li.find("strong")
                    if span and strong:
                        data[span.get_text(strip=True)] = strong.get_text(strip=True)
                nested_json = {"Interior": data}


                gernal_condition = json.dumps(nested_json, indent=4)
                row["General Condition"] = gernal_condition
                
                
                tyres_div = None
                for div in soup.find_all("div"):
                    strong_tag = div.find("strong")
                    if strong_tag and strong_tag.get_text(strip=True) == "Tyres":
                        tyres_div = div
                        break

                tyres_conditions = []
                if tyres_div:
                    ul = tyres_div.find("ul", class_="details-list")
                    if ul:
                        for li in ul.find_all("li"):
                            tyre_name_el = li.find("span")
                            strong_val = li.find("strong")
                            
                            tyre_name = tyre_name_el.get_text(strip=True) if tyre_name_el else "Unknown"
                            val = strong_val.get_text(strip=True) if strong_val else ""
                            
                            if not val:
                                val = ""
                            
                            tyres_conditions.append(f"{tyre_name}: {val}")

                tyres_csv_value = ", ".join(tyres_conditions)

                row['Tyres Condition'] = tyres_csv_value
                
                base_url = "https://www.protruckauctions.co.uk/"

                inspection_div = soup.find("div", class_="header-cta")
                inspection_pdf = ""
                if inspection_div:
                    a_tag = inspection_div.find("a", href=True)
                    if a_tag:
                        href = a_tag['href'].strip()
                        if href:
                            if href.startswith("http"):
                                inspection_pdf = href
                            else:
                                inspection_pdf = base_url.rstrip("/") + "/" + href.lstrip("/")

                row["Inspection Report"] = inspection_pdf
                
                images_div = soup.find("div", class_="gallery-items")
                images_urls = []

                if images_div:
                    for img_tag in images_div.find_all("img", src=True):
                        src = img_tag['src'].strip()
                        if src:
                            src = src.replace("---256-192.jpg", "---1024-768.jpg")
                            images_urls.append(src)


                row["Images"] = ", ".join(images_urls)
                damage_div = soup.find("div", class_="condition-gallery")
                damage_images = []
                damage_details = []

                if damage_div:
                    for figure in damage_div.find_all("figure", class_="condition-image"):
                        img_tag = figure.find("img", src=True)
                        if img_tag:
                            src = img_tag['src'].strip()
                            if src:
                                if src.startswith("/"):
                                    src = base_url + src
                                src = src.replace("---1140-855.jpg", "---1024-768.jpg")
                                damage_images.append(src)
                        figcaption = figure.find("figcaption")
                        if figcaption:
                            caption = " ".join(figcaption.get_text(" ", strip=True).split())
                            if caption:
                                damage_details.append(caption)

                row["Damaged_images"] = ", ".join(damage_images)
                row["Damage_details"] = ", ".join(damage_details)

                sheet_id, auction_name = get_base_folder_info()
                row["Sheet id"] = sheet_id or ""
                row["Auction Name"] = auction_name or ""
                row["Auction House"] =  "ProTruck"
                
                
                all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=keys)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()



✔ CSV Generated: protruck_data.csv


In [4]:
import os
import threading
import requests
import pandas as pd
from urllib.parse import urlparse, urljoin
from PIL import Image, ImageDraw, ImageFont


# ----------------------------------------
# IMAGE WATERMARK
# ----------------------------------------
def add_watermark_to_image(image_path, text="Sourced from ProTruck"):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

        try:
            font = ImageFont.truetype("arial.ttf", 18)
        except:
            font = ImageFont.load_default()

        margin = 10
        bbox = draw.textbbox((0, 0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]

        x = image.width - text_w - margin
        y = image.height - text_h - margin

        draw.rectangle(
            [x - margin, y - margin, x + text_w + margin, y + text_h + margin],
            fill=(0, 0, 0, 160)
        )
        draw.text((x, y), text, font=font, fill=(255, 255, 255, 255))

        final = Image.alpha_composite(image, txt_layer).convert("RGB")
        final.save(image_path)

        print(f"Watermark added: {image_path}")
    except Exception as e:
        print(f"Watermark fail {image_path}: {e}")


# ----------------------------------------
# LOAD CSV
# ----------------------------------------
df = pd.read_csv("protruck_data.csv")
reg_img = df[['Reg', 'Images']]
cond_imgs = df[['Damage_details', 'Damaged_images', 'Reg']]


# ----------------------------------------
# DOWNLOAD NORMAL IMAGES
# ----------------------------------------
def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg = row["Reg"]
        images = row["Images"]

        if pd.isna(images) or not str(images).strip():
            print(f"No images for {reg}")
            continue

        urls = images.split(", ")
        reg_folder = os.path.join(main_folder, reg)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, url in enumerate(urls):
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
                save_path = os.path.join(reg_folder, f"{reg}_{idx+1}{ext}")

                with open(save_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(save_path)
                print(f"Downloaded: {save_path}")

            except Exception as e:
                print(f"Image failed {url}: {e}")


# ----------------------------------------
# DOWNLOAD DAMAGE IMAGES
# ----------------------------------------
def download_images_damage(data, main_folder="Damage_203"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg = row["Reg"]
        dmg_imgs = row["Damaged_images"]
        dmg_texts = row["Damage_details"]

        if pd.isna(dmg_imgs) or pd.isna(dmg_texts):
            print(f"No damaged images for {reg}")
            continue

        urls = dmg_imgs.split(", ")
        texts = dmg_texts.split(", ")

        reg_folder = os.path.join(main_folder, reg)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, (url, text) in enumerate(zip(urls, texts)):
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            safe_text = "".join(c if c.isalnum() or c in " _-" else "_" for c in text)

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
                save_path = os.path.join(reg_folder, f"{safe_text}_{idx+1}{ext}")

                with open(save_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(save_path)
                print(f"Damage downloaded: {save_path}")

            except Exception as e:
                print(f"Damage failed {url}: {e}")


# ----------------------------------------
# RUN THREADS
# ----------------------------------------
def start_funcs():
    threads = [
        threading.Thread(target=download_images, args=(reg_img,)),
        threading.Thread(target=download_images_damage, args=(cond_imgs,))
    ]

    for t in threads:
        t.start()

    for t in threads:
        t.join()


if __name__ == "__main__":
    start_funcs()


Watermark added: Images\AV64RZF\AV64RZF_1.jpg
Downloaded: Images\AV64RZF\AV64RZF_1.jpg
Watermark added: Damage_203\AV64RZF\Panel Rear Dented Over 30_ Of Panel_1.jpg
Damage downloaded: Damage_203\AV64RZF\Panel Rear Dented Over 30_ Of Panel_1.jpg
Watermark added: Images\AV64RZF\AV64RZF_2.jpg
Downloaded: Images\AV64RZF\AV64RZF_2.jpg
Watermark added: Damage_203\AV64RZF\Panel Rear Dented Over 30_ Of Panel_2.jpg
Damage downloaded: Damage_203\AV64RZF\Panel Rear Dented Over 30_ Of Panel_2.jpg
Watermark added: Images\AV64RZF\AV64RZF_3.jpg
Downloaded: Images\AV64RZF\AV64RZF_3.jpg
Watermark added: Images\AV64RZF\AV64RZF_4.jpg
Downloaded: Images\AV64RZF\AV64RZF_4.jpg
Watermark added: Damage_203\AV64RZF\Door osf Dented With Paint Damage_3.jpg
Damage downloaded: Damage_203\AV64RZF\Door osf Dented With Paint Damage_3.jpg
Watermark added: Images\AV64RZF\AV64RZF_5.jpg
Downloaded: Images\AV64RZF\AV64RZF_5.jpg
Watermark added: Images\AV64RZF\AV64RZF_6.jpg
Downloaded: Images\AV64RZF\AV64RZF_6.jpg
Watermar

In [5]:
import os
import time
import pandas as pd
import requests
import json

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


EMAIL = "fourbrotherstrading@icloud.com"
PASSWORD = "Muhssan7865"


def setup_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver


# -------------------------------
# LOGIN
# -------------------------------
def login(driver):
    driver.get("https://www.protruckauctions.co.uk/login")
    wait = WebDriverWait(driver, 10)

    wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys(EMAIL)
    driver.find_element(By.ID, "password").send_keys(PASSWORD)
    driver.find_element(By.ID, "sign-in").click()

    time.sleep(2)
    print("[+] Login successful!")


# -------------------------------
# EXPORT COOKIES FOR REQUESTS
# -------------------------------
def get_session_cookies(driver):
    cookies = driver.get_cookies()
    session = requests.Session()

    for cookie in cookies:
        session.cookies.set(cookie['name'], cookie['value'])

    return session


# -------------------------------
# DOWNLOAD PDF using REAL session
# -------------------------------
def download_pdf(session, url, save_path):
    print(f"[+] Downloading: {url}")

    response = session.get(url)
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)

        print(f"[✔] Saved: {save_path}")
    else:
        print(f"[X] Failed ({response.status_code}) : {url}")


# -------------------------------
# MAIN
# -------------------------------
def download_all_pdfs(csv_file):
    df = pd.read_csv(csv_file)

    base_folder = "Inspection Reports"
    os.makedirs(base_folder, exist_ok=True)

    driver = setup_driver()
    login(driver)

    session = get_session_cookies(driver)   # ❤️ MAGIC LINE
    driver.quit()

    for _, row in df.iterrows():
        reg = row["Reg"]
        url = row["Inspection Report"]

        if pd.isna(url) or not str(url).strip():
            print(f"[!] Missing PDF for {reg}")
            continue

        save_path = os.path.join(base_folder, f"{reg}.pdf")
        download_pdf(session, url, save_path)

    print("\nAll PDFs downloaded!")


# -------------------------------
if __name__ == "__main__":
    download_all_pdfs("protruck_data.csv")


[+] Login successful!
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/35083.pdf
[✔] Saved: Inspection Reports\AV64RZF.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/52396.pdf
[✔] Saved: Inspection Reports\BF71XYZ.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/51902.pdf
[✔] Saved: Inspection Reports\BJ21NBM.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/51551.pdf
[✔] Saved: Inspection Reports\BJ22YON.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/49607.pdf
[✔] Saved: Inspection Reports\BK23XDF.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/52795.pdf
[✔] Saved: Inspection Reports\BL22ONO.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/47799.pdf
[✔] Saved: Inspection Repo

In [6]:
import os
from PyPDF2 import PdfReader, PdfWriter, Transformation
from reportlab.pdfgen import canvas

HEADER_HEIGHT = 25  


def create_header_page(text, filename, page_width, page_height):
    c = canvas.Canvas(filename, pagesize=(page_width, page_height))

    # HEX #047AFA → RGB normalized
    r, g, b = (4/255, 122/255, 250/255)

    # Background bar
    c.setFillColorRGB(r, g, b)
    c.rect(0, page_height - HEADER_HEIGHT, page_width, HEADER_HEIGHT, fill=1)

    # White centered text
    c.setFillColorRGB(1, 1, 1)
    c.setFont("Helvetica-Bold", 12)

    text_width = c.stringWidth(text, "Helvetica-Bold", 12)
    x = (page_width - text_width) / 2
    y = page_height - HEADER_HEIGHT + 7

    c.drawString(x, y, text)
    c.save()


def add_header_to_pdf(input_pdf, output_pdf):
    reader = PdfReader(input_pdf)
    writer = PdfWriter()

    for page in reader.pages:
        page_width = float(page.mediabox.width)
        page_height = float(page.mediabox.height)

        shift = Transformation().translate(0, -HEADER_HEIGHT)
        page.add_transformation(shift)

        temp_header = "header_temp.pdf"
        create_header_page("Source from Protruck Auctions", temp_header, page_width, page_height)

        header_pdf = PdfReader(temp_header)
        header_page = header_pdf.pages[0]

        page.merge_page(header_page)
        writer.add_page(page)

    with open(output_pdf, "wb") as f:
        writer.write(f)

    os.remove(temp_header)


def add_header_to_all_pdfs(folder):
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            input_pdf = os.path.join(folder, file)
            output_pdf = os.path.join(folder, file)

            print(f"Fixing & adding new header to: {file}")
            add_header_to_pdf(input_pdf, output_pdf)

    print("\n✔ All PDFs updated with blue header (#047AFA)!")


if __name__ == "__main__":
    add_header_to_all_pdfs("Inspection Reports")


Fixing & adding new header to: AV64RZF.pdf
Fixing & adding new header to: BF71XYZ.pdf
Fixing & adding new header to: BJ21NBM.pdf
Fixing & adding new header to: BJ22YON.pdf
Fixing & adding new header to: BK23XDF.pdf
Fixing & adding new header to: BL22ONO.pdf
Fixing & adding new header to: BT70UAS.pdf
Fixing & adding new header to: CE70LTV.pdf
Fixing & adding new header to: DE21VJL.pdf
Fixing & adding new header to: DE21VLG.pdf
Fixing & adding new header to: DX23CFL.pdf
Fixing & adding new header to: FD65ORA.pdf
Fixing & adding new header to: FG71RMZ.pdf
Fixing & adding new header to: FJ20UZW.pdf
Fixing & adding new header to: FL18ENO.pdf
Fixing & adding new header to: FL65RCO.pdf
Fixing & adding new header to: FP65YUO.pdf
Fixing & adding new header to: FY24EOS.pdf
Fixing & adding new header to: HV11FMF.pdf
Fixing & adding new header to: HV24ZYA.pdf
Fixing & adding new header to: HV74KGU.pdf
Fixing & adding new header to: HY18TSU.pdf
Fixing & adding new header to: KX13JXC.pdf
Fixing & ad